In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

trips = spark.read.table("samples.nyctaxi.trips")
display(trips.limit(10))

In [0]:
trips.printSchema()
trips.select(
    F.count("*").alias("total_rows"),
    F.min("fare_amount").alias("min_fare"),
    F.max("fare_amount").alias("max_fare"),
    F.avg("trip_distance").alias("avg_distance")
).show()

In [0]:
clean = (
    trips
    .filter(F.col("trip_distance") > 0)
    .filter(F.col("fare_amount") > 0)
    .withColumn("trip_duration_min", (F.col("tpep_dropoff_datetime").cast("long") - F.col("tpep_pickup_datetime").cast("long")) / 60)
    .withColumn("speed_mph", F.col("trip_distance") / (F.col("trip_duration_min") / 60))
    .withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
)
display(clean.limit(10))

In [0]:
daily = (
    clean
    .groupBy("pickup_date")
    .agg(
        F.count("*").alias("trip_count"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
        F.round(F.avg("trip_distance"), 2).alias("avg_distance"),
        F.round(F.avg("trip_duration_min"), 2).alias("avg_duration_min")
    )
    .orderBy("pickup_date")
)

display(daily)

In [0]:
clean.createOrReplaceTempView("clean_trips")

sql_result = spark.sql("""
SELECT
  pickup_date,
  pickup_hour,
  COUNT(*) AS trip_count,
  ROUND(AVG(fare_amount), 2) AS avg_fare
FROM clean_trips
GROUP BY pickup_date, pickup_hour
ORDER BY pickup_date, pickup_hour
""")

display(sql_result)

In [0]:
hourly = (
    clean.groupBy("pickup_hour")
    .agg(F.count("*").alias("trip_count"))
    .orderBy(F.desc("trip_count"))
)

display(hourly)

In [0]:
daily.write.mode("overwrite").format("delta").saveAsTable("workspace.default.taxi_daily_summary")

In [0]:
daily.show()